# 09｜独立评估统计与固定门槛

对应[第 09 章](../course/09-independent-evaluation.md)。本 Notebook 复核已提交的真实 compact evidence；它不会恢复策略，也不产生新的正式评估。

## 学习目标

从 successes/episodes 重算点估计；理解 Wilson 区间；检查 per-seed 加总和 worst seed；严格执行预注册门槛；区分随机评估样本与精选轨迹样例。

## 先预测

1. 964/1,024 是否达到固定 95% 点估计门槛？
2. 若 Wilson 上界超过 95%，工程门槛能否改判 PASS？
3. 四个 seed 分别成功 235、242、245、242，worst-seed rate 是多少？
4. 课程 fixture 有意选 4 成功、4 失败，能否报告策略成功率为 50%？

## 运行与观察

读取两类证据：compact report 保存完整 1,024 回合聚合；fixture 只保存 8 条用于学习轨迹字段的精选例子。

In [ ]:
from pathlib import Path
import math
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
from course_utils import assert_course_kernel, load_json, wilson_interval

assert_course_kernel(ROOT)
report_path = ROOT / 'reproduction/results/linux-guide-free-left-trajectory-analysis.json'
fixture_path = ROOT / 'docs/data/guide-free-left-episodes-fixture.json'
report = load_json(report_path)
fixture = load_json(fixture_path)
print('Formal evidence:', report_path.relative_to(ROOT))
print('Teaching fixture:', fixture_path.relative_to(ROOT))

### 1. 聚合点估计和 Wilson 区间

点估计描述这批样本；区间描述二项抽样不确定性；门槛是运行前冻结的决策规则。三者回答不同问题。

In [ ]:
episodes = report['protocol']['episodes']
successes = report['aggregate']['successes']
failures = report['aggregate']['failures']
rate = successes / episodes
interval = wilson_interval(successes, episodes)
print(f'successes = {successes}/{episodes} = {rate:.4%}')
print(f'Wilson 95% = [{interval[0]:.4%}, {interval[1]:.4%}]')
assert successes + failures == episodes
assert np.allclose(interval, report['aggregate']['success_rate_wilson_95'], atol=1e-6)

### 2. Per-seed 防止平均值隐藏弱批次

四个 held-out seed 的回合数相同，因此 aggregate 恰好也是四个 rate 的平均；一般情况下仍应按分子分母加总。

In [ ]:
per_seed = report['per_seed']
seed_labels = [str(row['seed']) for row in per_seed]
seed_rates = np.array([row['successes'] / row['episodes'] for row in per_seed])
worst_seed = float(seed_rates.min())
assert sum(row['successes'] for row in per_seed) == successes
assert sum(row['episodes'] for row in per_seed) == episodes
print('per-seed rates:', dict(zip(seed_labels, np.round(seed_rates, 4))))
print(f'worst seed = {worst_seed:.4%}')

plt.figure(figsize=(7, 3.5))
plt.bar(seed_labels, seed_rates, color='tab:blue')
plt.axhline(0.90, color='tab:green', linestyle='--', label='90% worst-seed gate')
plt.ylim(0.85, 1.0); plt.xlabel('held-out seed'); plt.ylabel('success rate')
plt.legend(); plt.title('Guide-free left evaluation by seed'); plt.show()

### 3. 样本量如何影响区间

在约 94% 点估计附近增加样本量，Wilson 区间通常收窄。62/64 看起来很高，但分母远小于 1,024。

In [ ]:
sample_sizes = np.array([32, 64, 128, 256, 512, 1024])
approx_successes = np.rint(sample_sizes * rate).astype(int)
sample_intervals = np.array([wilson_interval(int(k), int(n)) for k, n in zip(approx_successes, sample_sizes)])
widths = sample_intervals[:, 1] - sample_intervals[:, 0]
plt.figure(figsize=(7, 3.5))
plt.plot(sample_sizes, widths * 100, marker='o')
plt.xlabel('episodes'); plt.ylabel('Wilson interval width (percentage points)')
plt.title('More episodes reduce binomial uncertainty'); plt.grid(alpha=0.25); plt.show()
assert widths[-1] < widths[0]

### 4. 固定门槛不是置信区间

目标是 aggregate ≥95%、worst seed ≥90%。即使区间覆盖 95%，964/1,024 的点估计仍未达到预先规则。

In [ ]:
aggregate_target = 0.95
worst_seed_target = 0.90
required_successes = math.ceil(aggregate_target * episodes)
aggregate_passed = rate >= aggregate_target
worst_seed_passed = worst_seed >= worst_seed_target
print('aggregate:', 'PASS' if aggregate_passed else 'FAIL', f'(needs {required_successes - successes} more successes)')
print('worst seed:', 'PASS' if worst_seed_passed else 'FAIL')
print('overall:', 'PASS' if aggregate_passed and worst_seed_passed else 'FAIL')

## 动手修改

将 `hypothetical_successes` 改成 964、972、973、990。每次先预测 point-estimate gate，再运行。注意这是假设计算，不得覆盖正式 report。最后恢复 973。

In [ ]:
hypothetical_successes = 973
hypothetical_rate = hypothetical_successes / episodes
hypothetical_interval = wilson_interval(hypothetical_successes, episodes)
print(f'{hypothetical_successes}/{episodes} = {hypothetical_rate:.4%}')
print(f'Wilson 95% = [{hypothetical_interval[0]:.4%}, {hypothetical_interval[1]:.4%}]')
print('fixed gate:', 'PASS' if hypothetical_rate >= aggregate_target else 'FAIL')

## 自测

fixture 的 8 条记录被按每 seed 一成功一失败挑选，只用于查看字段。下面的断言故意阻止你把它当随机成功率样本。

In [ ]:
examples = fixture['curated_episode_examples']
assert fixture['fixture_role'] == 'teaching_only_curated_examples_not_a_rate_sample'
assert len(examples) == 8 and sum(row['success'] for row in examples) == 4
assert required_successes == 973 and required_successes - successes == 9
assert not aggregate_passed and worst_seed_passed
assert hypothetical_successes == required_successes and hypothetical_rate >= aggregate_target
print('PASS: aggregation, Wilson interval, per-seed audit, fixed gate, and fixture boundary')

## 反思与记录

用[评估结论模板](../templates/evaluation-conclusion.md)写四句话，必须包含 964/1,024、点估计、Wilson、worst seed 和固定门槛 FAIL。再说明：

- 本 Notebook 证明你会复核统计，但为什么最多是 Gate 4 PRACTICED？
- 要达到 READY，还缺自己的 checkpoint、完整 episode records、协议和 SHA-256 中的哪些证据？